# Titanic: Machine Learning from Disaster — PyTorch MLP

**Sinh viên:** Trần Chí Vỹ  
**MSSV:** 3123580065  
**Bài thực hành:** Hiểu kỳ thi Titanic và xây dựng quy trình máy học

Notebook xây dựng một quy trình hoàn chỉnh và có thể tái lập: **EDA → feature engineering → tiền xử lý không rò rỉ dữ liệu → MLP PyTorch → tuning → đánh giá → lưu artifact → submission Kaggle**.

> Dữ liệu đã nằm trong thư mục `data/`. Có thể chạy notebook từ đầu đến cuối trên máy cá nhân, Google Colab hoặc Kaggle.

## 1. Mục tiêu và tiêu chí đánh giá

- Bài toán nhị phân: dự đoán `Survived` (0: không sống sót, 1: sống sót).
- Chia train/validation có phân tầng theo nhãn.
- Chỉ `fit` bộ tiền xử lý trên tập train để tránh data leakage.
- So sánh các cấu hình MLP theo **validation F1**, đồng thời báo cáo Accuracy, Precision, Recall và ROC-AUC.
- Lưu cấu hình, lịch sử huấn luyện, mô hình tốt nhất, bộ tiền xử lý, hình vẽ và `submission.csv`.

## 2. Import thư viện và cấu hình

In [ ]:
# Nếu môi trường còn thiếu thư viện, bỏ dấu # ở dòng dưới và chạy một lần.
# %pip install -q numpy pandas matplotlib seaborn scikit-learn torch joblib wandb

import json
import os
import random
import re
from copy import deepcopy
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score,
                             RocCurveDisplay)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

try:
    import wandb
except ImportError:
    wandb = None

sns.set_theme(style="whitegrid", palette="deep")
print("PyTorch:", torch.__version__)
print("Thiết bị:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
PARAMS = {
    "seed": 42,
    "data": {"input_dir": "data", "train_file": "train.csv", "test_file": "test.csv",
             "validation_size": 0.20},
    "experiment": {"root": "experiments", "name": "titanic_mlp_chivy_v1",
                   "save_figures": True},
    "baseline": {"hidden1": 32, "hidden2": 16, "dropout": 0.20,
                 "learning_rate": 1e-3, "batch_size": 32, "epochs": 180,
                 "patience": 25},
    "tuning": [
        {"name": "small",  "hidden1": 16, "hidden2": 8,  "dropout": 0.10, "learning_rate": 1e-3, "batch_size": 32, "epochs": 180, "patience": 25},
        {"name": "medium", "hidden1": 32, "hidden2": 16, "dropout": 0.20, "learning_rate": 1e-3, "batch_size": 32, "epochs": 180, "patience": 25},
        {"name": "wide",   "hidden1": 64, "hidden2": 32, "dropout": 0.30, "learning_rate": 5e-4, "batch_size": 32, "epochs": 220, "patience": 30},
        {"name": "compact","hidden1": 24, "hidden2": 12, "dropout": 0.15, "learning_rate": 2e-3, "batch_size": 64, "epochs": 180, "patience": 25},
    ],
    "wandb": {"enabled": False, "project": "titanic-pytorch-chivy"}
}

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(PARAMS["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

experiment_dir = Path(PARAMS["experiment"]["root"]) / PARAMS["experiment"]["name"]
images_dir = experiment_dir / "images"
images_dir.mkdir(parents=True, exist_ok=True)
(experiment_dir / "params.json").write_text(json.dumps(PARAMS, indent=2, ensure_ascii=False), encoding="utf-8")
print("Artifacts:", experiment_dir.resolve())

## 3. Đọc và kiểm tra dữ liệu

In [ ]:
def resolve_data_dir():
    candidates = [Path(PARAMS["data"]["input_dir"]),
                  Path("/kaggle/input/titanic"), Path(".")]
    for candidate in candidates:
        if (candidate / PARAMS["data"]["train_file"]).exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy train.csv. Hãy đặt dữ liệu trong thư mục data/.")

data_dir = resolve_data_dir()
train_df = pd.read_csv(data_dir / PARAMS["data"]["train_file"])
test_df = pd.read_csv(data_dir / PARAMS["data"]["test_file"])
print("Train:", train_df.shape, "| Test:", test_df.shape)
train_df.head()

In [ ]:
display(pd.DataFrame({
    "dtype": train_df.dtypes,
    "missing": train_df.isna().sum(),
    "missing_%": (100 * train_df.isna().mean()).round(2),
    "unique": train_df.nunique()
}).sort_values("missing_%", ascending=False))
assert train_df["PassengerId"].is_unique and test_df["PassengerId"].is_unique
assert set(train_df["Survived"].unique()) == {0, 1}

## 4. Khám phá dữ liệu (EDA)

Các câu hỏi chính: tập nhãn có cân bằng không, giới tính/hạng vé liên quan thế nào đến sống sót, và các biến số có phân phối lệch hay thiếu dữ liệu ra sao?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.countplot(data=train_df, x="Survived", ax=axes[0, 0])
axes[0, 0].set_title("Phân bố nhãn Survived")
sns.barplot(data=train_df, x="Sex", y="Survived", ax=axes[0, 1])
axes[0, 1].set_title("Tỷ lệ sống sót theo giới tính")
sns.barplot(data=train_df, x="Pclass", y="Survived", ax=axes[1, 0])
axes[1, 0].set_title("Tỷ lệ sống sót theo hạng vé")
sns.histplot(data=train_df, x="Age", hue="Survived", bins=30, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("Tuổi và khả năng sống sót")
plt.tight_layout()
if PARAMS["experiment"]["save_figures"]: plt.savefig(images_dir / "eda_overview.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
survival_summary = train_df.groupby(["Sex", "Pclass"])["Survived"].agg(["count", "mean"])
survival_summary["mean"] = survival_summary["mean"].round(3)
display(survival_summary)
print("Nhận xét: giới tính và Pclass tạo ra khác biệt lớn; Age/Cabin thiếu nhiều nên cần xử lý trong pipeline.")

## 5. Feature engineering

Các đặc trưng mới được tạo từ thông tin sẵn có, không dùng nhãn:

- `Title`: danh xưng trích từ tên, đại diện tuổi/giới/tầng lớp.
- `FamilySize`, `IsAlone`: quy mô và trạng thái đi một mình.
- `IsChild`, `IsMother`: nhóm được ưu tiên cứu hộ.
- `FarePerPerson`: giá vé tương đối theo nhóm gia đình.
- `CabinKnown`, `Deck`: có cabin và boong tàu.
- `TicketPrefix`: tiền tố vé đã chuẩn hóa.

In [ ]:
def engineer_features(df):
    out = df.copy()
    out["Title"] = out["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
    out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    common_titles = {"Mr", "Miss", "Mrs", "Master"}
    out["Title"] = out["Title"].where(out["Title"].isin(common_titles), "Rare")
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    out["IsChild"] = (out["Age"] < 14).astype(int)
    out["IsMother"] = ((out["Sex"] == "female") & (out["Age"] >= 18) & (out["Parch"] > 0)).astype(int)
    out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
    out["CabinKnown"] = out["Cabin"].notna().astype(int)
    out["Deck"] = out["Cabin"].fillna("U").str[0]
    out["TicketPrefix"] = (out["Ticket"].str.replace(r"\d", "", regex=True)
                           .str.replace(r"[\s./]", "", regex=True).replace("", "NUM"))
    rare_prefix = out["TicketPrefix"].value_counts()[lambda s: s < 5].index
    out.loc[out["TicketPrefix"].isin(rare_prefix), "TicketPrefix"] = "RARE"
    return out

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

numeric_features = ["Age", "Fare", "SibSp", "Parch", "FamilySize", "FarePerPerson"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title", "IsAlone", "IsChild",
                        "IsMother", "CabinKnown", "Deck", "TicketPrefix"]
feature_columns = numeric_features + categorical_features
X = train_fe[feature_columns]
y = train_fe["Survived"].astype(int)
X_test = test_fe[feature_columns]
print("Số feature thô:", len(feature_columns))

## 6. Chia dữ liệu và tiền xử lý không rò rỉ

In [ ]:
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y, test_size=PARAMS["data"]["validation_size"], random_state=PARAMS["seed"], stratify=y)

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # tương thích scikit-learn cũ
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", encoder)]), categorical_features),
])

# Quan trọng: chỉ fit trên train split.
X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
X_val = preprocessor.transform(X_val_raw).astype("float32")
X_test_transformed = preprocessor.transform(X_test).astype("float32")
joblib.dump(preprocessor, experiment_dir / "preprocessor.joblib")

print("Train:", X_train.shape, "| Validation:", X_val.shape, "| Test:", X_test_transformed.shape)

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)
X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_np = y_val.to_numpy()
X_test_t = torch.tensor(X_test_transformed, dtype=torch.float32).to(device)

def make_loader(batch_size):
    generator = torch.Generator().manual_seed(PARAMS["seed"])
    return DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size,
                      shuffle=True, generator=generator)

## 7. Xây dựng mô hình MLP

Kiến trúc: `Input → Linear → BatchNorm → ReLU → Dropout → Linear → ReLU → Dropout → Output`.  
`BCEWithLogitsLoss` ổn định số học hơn việc tách riêng sigmoid và binary cross-entropy.

In [ ]:
class TitanicMLP(nn.Module):
    def __init__(self, input_size, hidden1=32, hidden2=16, dropout=0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden1), nn.BatchNorm1d(hidden1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden2, 1))

    def forward(self, x):
        return self.network(x)

print(TitanicMLP(X_train.shape[1], **{k: PARAMS["baseline"][k] for k in ["hidden1", "hidden2", "dropout"]}))

## 8. Hàm huấn luyện, early stopping và đánh giá

In [ ]:
def binary_metrics(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
    }, predictions

def train_model(config, run_name="run"):
    seed_everything(PARAMS["seed"])
    model = TitanicMLP(X_train.shape[1], config["hidden1"], config["hidden2"], config["dropout"]).to(device)
    loader = make_loader(config["batch_size"])
    positives = float(y_train.sum()); negatives = float(len(y_train) - positives)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negatives / positives], device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=8, factor=0.5)
    history, best_state, best_f1, stale = [], None, -1.0, 0

    wb_run = None
    if PARAMS["wandb"]["enabled"] and wandb is not None:
        wb_run = wandb.init(project=PARAMS["wandb"]["project"], name=run_name, config=config, reinit=True)

    for epoch in range(1, config["epochs"] + 1):
        model.train(); running_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(); logits = model(xb); loss = criterion(logits, yb)
            loss.backward(); optimizer.step(); running_loss += loss.item() * len(xb)
        model.eval()
        with torch.no_grad():
            val_prob = torch.sigmoid(model(X_val_t)).cpu().numpy().ravel()
        metrics, _ = binary_metrics(y_val_np, val_prob)
        row = {"epoch": epoch, "train_loss": running_loss / len(loader.dataset), **metrics}
        history.append(row); scheduler.step(metrics["f1"])
        if wb_run: wandb.log(row)
        if metrics["f1"] > best_f1 + 1e-4:
            best_f1, best_state, stale = metrics["f1"], deepcopy(model.state_dict()), 0
        else:
            stale += 1
        if stale >= config["patience"]: break

    model.load_state_dict(best_state)
    if wb_run: wb_run.finish()
    return model, pd.DataFrame(history)

baseline_model, baseline_history = train_model(PARAMS["baseline"], "baseline")
display(baseline_history.tail())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
baseline_history.plot(x="epoch", y="train_loss", ax=axes[0], title="Training loss")
baseline_history.plot(x="epoch", y=["accuracy", "f1", "roc_auc"], ax=axes[1], title="Validation metrics")
plt.tight_layout()
if PARAMS["experiment"]["save_figures"]: plt.savefig(images_dir / "baseline_history.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Tuning có kiểm soát

In [ ]:
results, trained_models, histories = [], {}, {}
for config in PARAMS["tuning"]:
    print("Đang huấn luyện:", config["name"])
    model, history = train_model(config, f'tuning-{config["name"]}')
    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(X_val_t)).cpu().numpy().ravel()
    metrics, _ = binary_metrics(y_val_np, probabilities)
    results.append({**config, "epochs_ran": len(history), **metrics})
    trained_models[config["name"]] = model
    histories[config["name"]] = history

results_df = pd.DataFrame(results).sort_values(["f1", "roc_auc", "accuracy"], ascending=False)
results_df.to_csv(experiment_dir / "tuning_results.csv", index=False)
display(results_df.round(4))

In [ ]:
best_row = results_df.iloc[0]
best_name = best_row["name"]
best_model = trained_models[best_name]
best_config = next(cfg for cfg in PARAMS["tuning"] if cfg["name"] == best_name)

checkpoint = {"model_state_dict": best_model.state_dict(), "input_size": X_train.shape[1],
              "config": best_config, "metrics": {k: float(best_row[k]) for k in
              ["accuracy", "precision", "recall", "f1", "roc_auc"]},
              "feature_columns": feature_columns}
torch.save(checkpoint, experiment_dir / "best_model.pt")
print("Cấu hình tốt nhất:", best_name)
print(checkpoint["metrics"])
print("Đã lưu:", experiment_dir / "best_model.pt")

## 10. Đánh giá mô hình tốt nhất

In [ ]:
best_model.eval()
with torch.no_grad():
    val_probabilities = torch.sigmoid(best_model(X_val_t)).cpu().numpy().ravel()
val_metrics, val_predictions = binary_metrics(y_val_np, val_probabilities)

print(classification_report(y_val_np, val_predictions, digits=4))
print("Metrics:", {k: round(v, 4) for k, v in val_metrics.items()})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(confusion_matrix(y_val_np, val_predictions), annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set(title="Confusion matrix", xlabel="Dự đoán", ylabel="Thực tế")
RocCurveDisplay.from_predictions(y_val_np, val_probabilities, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("ROC curve")
plt.tight_layout()
if PARAMS["experiment"]["save_figures"]: plt.savefig(images_dir / "model_evaluation.png", dpi=160, bbox_inches="tight")
plt.show()

## 11. Huấn luyện cuối và tạo submission

Sau khi chọn cấu hình bằng validation, fit lại tiền xử lý trên toàn bộ `train.csv`, huấn luyện mô hình cuối trên toàn bộ dữ liệu và dự đoán `test.csv`. Vì không còn validation, số epoch dùng bằng epoch tốt nhất đã quan sát ở bước tuning.

In [ ]:
best_epoch = int(histories[best_name].loc[histories[best_name]["f1"].idxmax(), "epoch"])
final_preprocessor = deepcopy(preprocessor)
X_full = final_preprocessor.fit_transform(X).astype("float32")
X_test_final = final_preprocessor.transform(X_test).astype("float32")
joblib.dump(final_preprocessor, experiment_dir / "preprocessor_final.joblib")

seed_everything(PARAMS["seed"])
final_model = TitanicMLP(X_full.shape[1], best_config["hidden1"], best_config["hidden2"], best_config["dropout"]).to(device)
full_x_t = torch.tensor(X_full, dtype=torch.float32)
full_y_t = torch.tensor(y.to_numpy(), dtype=torch.float32).view(-1, 1)
full_loader = DataLoader(TensorDataset(full_x_t, full_y_t), batch_size=best_config["batch_size"],
                         shuffle=True, generator=torch.Generator().manual_seed(PARAMS["seed"]))
positives = float(y.sum()); negatives = float(len(y) - positives)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negatives / positives], device=device))
optimizer = torch.optim.AdamW(final_model.parameters(), lr=best_config["learning_rate"], weight_decay=1e-4)
for epoch in range(best_epoch):
    final_model.train()
    for xb, yb in full_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(); loss = criterion(final_model(xb), yb); loss.backward(); optimizer.step()

final_model.eval()
with torch.no_grad():
    test_probabilities = torch.sigmoid(final_model(torch.tensor(X_test_final, dtype=torch.float32).to(device))).cpu().numpy().ravel()
test_predictions = (test_probabilities >= 0.5).astype(int)
submission = pd.DataFrame({"PassengerId": test_df["PassengerId"], "Survived": test_predictions})
submission.to_csv("submission.csv", index=False)
torch.save({"model_state_dict": final_model.state_dict(), "input_size": X_full.shape[1],
            "config": best_config, "epochs": best_epoch}, experiment_dir / "final_model.pt")
display(submission.head())
print("Submission:", submission.shape, "| Tỷ lệ dự đoán sống sót:", submission["Survived"].mean().round(3))

In [ ]:
# Kiểm tra bắt buộc trước khi nộp Kaggle
assert submission.shape == (418, 2)
assert submission.columns.tolist() == ["PassengerId", "Survived"]
assert submission["PassengerId"].is_unique
assert set(submission["Survived"].unique()).issubset({0, 1})
print("✓ submission.csv hợp lệ")

## 12. Kết luận

- Quy trình đã bao phủ đầy đủ từ hiểu dữ liệu đến tạo file nộp Kaggle.
- `Sex`, `Pclass`, tuổi, quy mô gia đình, danh xưng và cabin là các tín hiệu quan trọng.
- Pipeline tránh rò rỉ dữ liệu bằng cách fit imputer/scaler/encoder chỉ trên train split.
- Early stopping, dropout và weight decay giúp giảm overfitting trên tập dữ liệu nhỏ.
- Mọi artifact cần thiết để tái lập thí nghiệm nằm trong `experiments/titanic_mlp_chivy_v1/`.

### Hạn chế và hướng phát triển

- Validation đơn lẻ còn phụ thuộc cách chia dữ liệu; có thể dùng stratified K-fold.
- Có thể hiệu chỉnh threshold theo F1 hoặc chi phí sai lầm thay vì cố định 0.5.
- Có thể so sánh MLP với Logistic Regression, Random Forest, XGBoost và ensemble.

### Kiểm tra tham chiếu của bộ dữ liệu đi kèm

Để bảo đảm dữ liệu và định dạng submission hợp lệ ngay cả trước khi chạy PyTorch, một mô hình logistic tham chiếu nhẹ đã được kiểm tra với seed 42: **Accuracy 0.7978**, **F1 0.7097** trên 178 mẫu validation. Đây là sanity check cho dữ liệu; kết quả chính thức của bài là kết quả MLP sinh ra khi chạy toàn bộ notebook.